# Running photoD on Rubin DP2

A clone of this branch has everything the fit needs: the locus, the TRILEGAL prior maps and the 3D dust
curves are all in `data/`. Nothing has to be built or downloaded first.

    python scripts/run_dp2.py --catalog <rubin_dp2/object_collection> --out <dir> --name dp2_photod --workers 6

This notebook does that on one field first, checks the answer, and then runs the survey.

## 1. What a clone gives you

In [ ]:
import os, subprocess, sys, glob, time
from pathlib import Path

photod  = Path("..").resolve()          # the checkout
catalog = "/astro/store/shire/hats/catalogs/rubin_dp2/object_collection"
out     = "/tmp/photod"                 # where the answers go

for name in ("data/LSSTlocus_10Gyr_DP2.txt", "data/priors_dp2.npz", "data/dust_dp2.npz"):
    f = photod / name
    print(f"{name:32s} {'ok' if f.exists() else 'MISSING':>8}  {f.stat().st_size / 2**20:7.1f} MB")

## 2. One field first

Always worth doing before committing to the survey: it takes a couple of minutes and catches a wrong path or
a missing column straight away. `--cone RA DEC RADIUS` takes a piece of sky.

In [ ]:
%%time
def run(*extra):
    command = [sys.executable, str(photod / "scripts/run_dp2.py"),
               "--catalog", catalog, "--out", out, *extra]
    print(" ".join(command[2:]), "\n")
    done = subprocess.run(command, text=True)
    assert done.returncode == 0, f"the run failed with {done.returncode}"

run("--name", "field", "--cone", "287.33", "-22.985", "0.4", "--workers", "4", "--overwrite")

## 3. Did every star come out the other side

The check worth keeping: the fit should answer for every star the selection produced. It is the one thing
that catches a step quietly dropping rows, which is easy to miss because the stars that survive look fine.

In [ ]:
import pyarrow.parquet as pq

def rows(path):
    return sum(pq.ParquetFile(f).metadata.num_rows
               for f in glob.glob(f"{path}/dataset/**/*.parquet", recursive=True))

print(f"{rows(f'{out}/field'):,} stars fitted")

## 4. What the fit gives you

Per star: the 14th, 50th and 86th percentiles of the absolute magnitude `Mr`, the metallicity `FeH`, the
extinction `Ar`, the reddened absolute magnitude `Qr` and the distance modulus `DM`, the entropy the data took
out of the prior for the first three, `chi2min`, and `flags`.

The distance is `10 ** (DM / 5 + 1)` parsecs. `DM` comes from the posterior of `Mr + Ar` rather than from
combining two marginals, so its percentiles are the right ones.

In [ ]:
import numpy as np

def readAll(path, columns):
    files = sorted(glob.glob(f"{path}/dataset/**/*.parquet", recursive=True))
    step = max(1, len(files) // 300)
    got = {c: [] for c in columns}
    for f in files[::step]:
        t = pq.read_table(f, columns=columns)
        for c in columns:
            got[c].append(t[c].to_numpy(zero_copy_only=False))
    return {c: np.concatenate(v) for c, v in got.items()}

cols = ["rmag", "chi2min", "flags", "DM_quantile_median", "Mr_quantile_median",
        "FeH_quantile_median", "Ar_quantile_median"]
d = readAll(f"{out}/field", cols)
for c in cols:
    v = d[c].astype(float)
    print(f"  {c:22s} 5/50/95 %: " + "  ".join(f"{x:8.3f}" for x in np.nanpercentile(v, [5, 50, 95])))

### The flags

One bit per thing worth knowing about the answer. Nothing is dropped for being flagged.

| bit | meaning |
|---|---|
| 1 | chi2 above 100, or no answer: the locus does not pass through this star's colours |
| 2 | the Mr posterior is lopsided, which is how a giant and a dwarf solution both survive |
| 4 | [Fe/H] is against the end of the model grid, so it is a limit rather than a measurement |
| 8 | A_r is against the top of its grid, and the distance goes wrong with it |
| 16 | a colour had no measurement and carried no weight |

Bit 16 is normal rather than a fault: it is nearly all the u band, which is the shallowest, and about half of
DP2 has no usable u-g. Against Gaia parallaxes the stars with bit 1 scatter five to twenty five times their
quoted uncertainty where the rest scatter 1.09 times it, so **`flags & 3 == 0` is the cut to start from**.

In [ ]:
flags = d["flags"].astype(int)
print(f"{len(flags):,} stars, usable at flags & 3 == 0: {100 * np.mean(flags & 3 == 0):.2f} %")
for bit, label in ((1, "poor fit or no answer"), (2, "two branches"), (4, "FeH at the grid end"),
                   (8, "A_r at the grid top"), (16, "a colour missing")):
    print(f"  bit {bit:2d} {label:22s}: {100 * np.mean((flags & bit) > 0):6.2f} %")

## 5. Does it look like the Galaxy

The cheapest check that the distances and the metallicities mean anything: take stars bright enough in
absolute terms to be seen across the whole range, so the sample is volume limited, and look at the
metallicity against height above the plane. It should run from the thin disc near the plane to the halo
several kiloparsecs up, and nothing in the fit was told to do that.

In [ ]:
from astropy.coordinates import SkyCoord

p = readAll(f"{out}/field", ["ra", "dec", "flags", "DM_quantile_median",
                             "Mr_quantile_median", "FeH_quantile_median"])
keep = (p["flags"].astype(int) & 3 == 0) & np.isfinite(p["DM_quantile_median"].astype(float))
keep &= (p["Mr_quantile_median"].astype(float) > 4) & (p["Mr_quantile_median"].astype(float) < 6)
b = SkyCoord(p["ra"][keep], p["dec"][keep], unit="deg").galactic.b.deg
z = np.abs(10 ** (p["DM_quantile_median"].astype(float)[keep] / 5 + 1) * np.sin(np.radians(b))) / 1000
feh = p["FeH_quantile_median"].astype(float)[keep]
print(f"{'|z| kpc':>12} {'stars':>8} {'[Fe/H] median':>15}")
for lo, hi in ((0.3, 1), (1, 2), (2, 4), (4, 9)):
    s = (z >= lo) & (z < hi)
    if s.sum() > 30:
        print(f"{lo:5.1f}-{hi:<6.1f} {int(s.sum()):8d} {np.median(feh[s]):15.3f}")

## 6. The whole survey

Same command without `--cone`. On two L40s it took 93 minutes for 93.7 million stars, about 16800 a second
end to end, and wrote 16 GB. `--workers` is processes, which share whatever GPUs are there; two per card is
a reasonable start. Memory stays flat because a worker is replaced every few hundred partitions.

Uncomment and run when you are ready.

In [ ]:
# run("--name", "dp2_photod", "--workers", "6", "--overwrite")

## 7. If your catalog is already prepared

A catalog that already carries `ra`, `dec`, `rmag`, the five colours and their errors is used as it is; the
fit only builds colours from PSF fluxes when it has to. Name the extinction column it came with:

    --ar-column Ar_SFD

## 8. Another footprint, or the exact maps

`scripts/make_priors.py` builds prior maps for a footprint out of TRILEGAL, `--compact` for the small form
that ships here, and `--from-catalog` converts maps that already exist as a HATS catalog.
`scripts/make_dust_curves.py` builds the dust curves, taking several 3D maps and letting the deepest with
data win each sightline.

The maps in `data/` are the compact form, which moves the median star by under two millimagnitudes. The exact
maps are attached to the release; pass them with `--priors` and nothing else changes.